# Lumis 01: Data setup

Sets up the Lumis folders in Google Drive, downloads DermNet from Kaggle, and counts images per class so the final class list can be locked.

Before running:
1. **Runtime > Change runtime type > T4 GPU**
2. Add your Kaggle API token as a Colab secret (key icon in the left sidebar): name it `KAGGLE_API_TOKEN`, paste the token (starts with `KGAT_`) as the value, and switch on notebook access. Never paste the token into a code cell, since this notebook is saved to GitHub.

## 1. Mount Google Drive and create the Lumis folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/Lumis'
for sub in ['data', 'checkpoints', 'exports']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)
print('Drive folders:', sorted(os.listdir(BASE)))

## 2. Check the GPU

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE - switch the runtime to T4 GPU')

## 3. Download DermNet into Drive (only the first time)

The zip is kept in Drive so it survives between sessions.

In [ ]:
import subprocess
from google.colab import userdata

# Newer Kaggle client reads the single KAGGLE_API_TOKEN
subprocess.run(['pip', 'install', '-q', '-U', 'kaggle'], check=True)
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

ZIP = f'{BASE}/data/dermnet.zip'
if os.path.exists(ZIP):
    print('Already downloaded:', ZIP)
else:
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'shubhamgoel27/dermnet',
                    '-p', f'{BASE}/data'], check=True)
    print('Downloaded:', os.listdir(f'{BASE}/data'))

## 4. Unzip to the Colab machine (fast local disk, redo each session)

In [ ]:
import zipfile
LOCAL = '/content/dermnet'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(LOCAL)
print(sorted(os.listdir(LOCAL)))

## 5. Count images per class

Kaggle's train/test split is ignored later: Lumis pools both and re-splits 70/15/15 (CP1 section 3.5.1). The `lumis_class` column shows which DermNet folders map to the five planned classes.

In [ ]:
import pandas as pd
from pathlib import Path

IMG_EXT = {'.jpg', '.jpeg', '.png'}
rows = []
for split_dir in sorted(p for p in Path(LOCAL).rglob('*') if p.is_dir() and p.name.lower() in ('train', 'test')):
    for cls in sorted(d for d in split_dir.iterdir() if d.is_dir()):
        n = sum(1 for f in cls.iterdir() if f.suffix.lower() in IMG_EXT)
        rows.append({'split': split_dir.name, 'class': cls.name, 'images': n})

counts = (pd.DataFrame(rows)
          .pivot_table(index='class', columns='split', values='images', aggfunc='sum', fill_value=0))
counts['total'] = counts.sum(axis=1)

TARGETS = {'eczema': 'eczema', 'atopic': 'eczema', 'tinea': 'tinea',
           'acne': 'acne', 'psoriasis': 'psoriasis', 'scabies': 'scabies'}
counts['lumis_class'] = [next((v for k, v in TARGETS.items() if k in c.lower()), '') for c in counts.index]
counts = counts.sort_values('total', ascending=False)
display(counts)

print('\nImages per planned Lumis class (before cleaning):')
print(counts[counts.lumis_class != ''].groupby('lumis_class')['total'].sum().sort_values())

counts.to_csv(f'{BASE}/data/dermnet_class_counts.csv')
print('\nSaved to Drive: Lumis/data/dermnet_class_counts.csv')